In [122]:
import pandas as pd
pd.set_option("display.width", 200)

In [123]:
##############
# Import CSV
##############
df = pd.read_csv("../data/netflix_titles.csv")
# test = pd.read_csv("../data/netflix_titles.csv")
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB
None


In [124]:
############
# Extracting important data
############

netflix_df = df[["show_id", "type", "title" ,"country", "date_added", "release_year", "rating", "duration", "listed_in"]]
netflix_df.head(5)

,show_id,type,title,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries
1,s2,TV Show,Blood & Water,South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries"
2,s3,TV Show,Ganglands,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
3,s4,TV Show,Jailbirds New Orleans,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV"
4,s5,TV Show,Kota Factory,India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ..."


In [125]:
###########
# Amount of nulls in each column
###########
netflix_df.isna().sum().sort_values(ascending=False)

country         831
date_added       10
rating            4
duration          3
show_id           0
type              0
title             0
release_year      0
listed_in         0
dtype: int64

In [126]:
###########
# Cleaning country column
###########
netflix_df["country"] = netflix_df["country"].fillna("Unknown")


In [127]:
###########
# Splitting the country column into a new table. Better for data modeling and to avoid duplicated rows 
###########
# 194 show_id and 366 ili 193 365
country_df = netflix_df[["show_id", "country"]].copy()

country_df['country'] = country_df['country'].str.split(',').apply(
    lambda x: [i.strip() for i in x if i.strip()]
)

country_df = country_df.explode("country")

# print(country_df.groupby("country").size().to_string())

# Deleted country column netflix_df
netflix_df = netflix_df.drop(columns=["country"])

In [128]:
###########
# Moving values from the rating to the duration columns
###########
dirty_rating = netflix_df["rating"].str.contains("min", na=False)

netflix_df.loc[dirty_rating, "duration"] = netflix_df.loc[dirty_rating, "rating"]

netflix_df.loc[dirty_rating, "rating"] = pd.NA

###########
# Extract numeric values from the duration column and changing type to int
###########

netflix_df["duration"] = netflix_df["duration"].str.extract(r"(\d+)")

netflix_df["duration"] = netflix_df["duration"].astype("Int64")

In [129]:
###########
# Cleaning rating column
###########
netflix_df["rating"] = netflix_df["rating"].fillna("Unrated")


In [130]:
###########
# Trimming column date_added and changing data type to datetime
###########
netflix_df["date_added"] = netflix_df["date_added"].str.strip()

netflix_df["date_added"] = pd.to_datetime(netflix_df["date_added"], format="%B %d, %Y", errors="coerce")
